# 04 — E2 AdminQA Fine-tuning (VNAB v1.0)
Ngày 4 • Chỉ train/dev. Không tải test VNAB, không đọc gold test.

- E2-M1: checkpoint gốc, không train.
- E2-M2: checkpoint gốc → VNAB train.
- E2-M3 (tùy chọn, mặc định tắt): E1 Epoch 2 → VNAB train. Thêm E1-only diagnostic để tách hiệu quả khởi tạo và fine-tuning VNAB.
- Chọn epoch theo F1 dev, EM, rồi epoch sớm hơn. Chọn cấu hình triển khai theo F1 dev, EM, rồi M1/M2/M3 theo thứ tự đơn giản hơn. Báo cáo tất cả cấu hình thực nghiệm, kể cả khi kém baseline.
- Metric giữ nguyên NFC/lowercase/whitespace, giữ dấu câu và dấu tiếng Việt. F1 theo token tách khoảng trắng. Không tuyên bố tương đương điểm của paper.
- Chỉ chạy tuần tự 01–09. Mục 10 là khóa sau khi xem dev, không chạy test.

## 01. Môi trường và GPU
Nếu phải cài thư viện, restart session và chạy lại 01. GPU T4 ưu tiên; không cài lại torch.

In [2]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
REQUIRED = {"transformers":"4.57.1", "datasets":"4.8.5", "huggingface-hub":"0.36.2", "accelerate":"1.15.0", "sentencepiece":"0.2.2"}
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
changes = [f"{k}=={v}" for k,v in REQUIRED.items() if installed(k)!=v]
changes += [k for k in ("pandas","protobuf","tqdm","jsonschema") if installed(k) is None]
if changes:
    print("Cần cài:", changes, flush=True)
    subprocess.check_call([sys.executable,"-m","pip","install",*changes])
    raise SystemExit("Cài xong. Restart session rồi chạy lại mục 01.")
import torch, transformers, datasets, pandas as pd, numpy as np
import json, re, unicodedata, time, math, shutil, random, hashlib, platform
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline, set_seed
from torch.utils.data import DataLoader
from IPython.display import display
SEED = 42
set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại 01. Không cài lại torch.")
DEVICE = torch.device("cuda:0")
print("GPU:",torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print("CUDA check:",(torch.ones(1,device=DEVICE)*2).item())
for k in REQUIRED: print(k,installed(k))
print("torch",torch.__version__)
print("ENVIRONMENT OK")
import zipfile, io, jsonschema, gc
from google.colab import files, drive

GPU: Tesla T4 VRAM GiB: 14.56
CUDA check: 2.0
transformers 4.57.1
datasets 4.8.5
huggingface-hub 0.36.2
accelerate 1.15.0
sentencepiece 0.2.2
torch 2.14.1+cu130
ENVIRONMENT OK


## 02. Cấu hình và lưu trên Drive
Mặc định chạy M1/M2; nếu muốn thêm M3, bật RUN_M3 trước khi chạy mục này. M3 dùng checkpoint E1 đã khóa, không dùng pilot. 3 epoch, cùng hyperparameter giữa M2/M3. Không dùng test để chọn tham số.

In [3]:
MODEL_NAME = "deepset/xlm-roberta-base-squad2"
MODEL_REVISION = "a5fab9908c8d856e8c583fd41ba6d92444e46477"
DATASET_SHA256 = "e1317a5d27d85426f1e22a0efdb695aacd335a77d79ae9ddd918dd86fa6651ee"
RUN_M3 = False
E1_DIR = Path("/content/drive/MyDrive/VNAB_research/E1_runs/full_20261127_143558/best_checkpoint")
E1_HASHES = {'config.json': '640cd8d980a283aeeb06c92f8680b7d5fc148e25ffa243b6700383ae9debf9c3', 'model.safetensors': 'fbfb34eac8c82ad39619e7d3f479c80680047e403a3ff8d36da7d736faa7b88c', 'sentencepiece.bpe.model': 'cfc8146abe2a0488e9e2a0c56de7952f7c11ab059eca145a0a727afce0db2865', 'special_tokens_map.json': '06e405a36dfe4b9604f484f6a1e619af1a7f7d09e34a8555eb0b77b66318067f', 'tokenizer.json': '9bed8d2b0662f3442135c4bff8fc9e20aa33347f4f8d1930d7a3f8c0cf8a85ac', 'tokenizer_config.json': '80496672bea865cc46e5a05581aca18059e546499dd401cbcfb7916a5a6540b7'}
EPOCHS = 3
BATCH_SIZE = 2
ACCUM_STEPS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
MAX_LENGTH = 384
STRIDE = 128
USE_FP16 = True
INIT_SCALE = 1024.0
QA_SETTINGS = {"max_seq_len":MAX_LENGTH,"doc_stride":STRIDE,"max_answer_len":128,"handle_impossible_answer":False}
SELECTION = "dev F1, then EM, then earliest epoch; method ties M1 before M2 before M3"
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/VNAB_research/E2_runs")
RUN_DIR = ROOT / ("vnab_"+time.strftime("%Y%m%d_%H%M%S"))
RUN_DIR.mkdir(parents=True,exist_ok=False)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
def save_json(name,obj):
    (RUN_DIR/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding="utf-8")
config = {"dataset":"VNAB v1.0","dataset_sha256":DATASET_SHA256,"model":MODEL_NAME,"model_revision":MODEL_REVISION,"run_m3":RUN_M3,"seed":SEED,"epochs":EPOCHS,"batch_size":BATCH_SIZE,"gradient_accumulation":ACCUM_STEPS,"learning_rate":LR,"weight_decay":WEIGHT_DECAY,"optimizer":"AdamW constant LR; clip grad norm 1.0","fp16":USE_FP16,"grad_scaler_init_scale":INIT_SCALE,"qa_settings":QA_SETTINGS,"selection":SELECTION,"metric_normalization":"NFC, lowercase, whitespace; retain punctuation and accents","unicode_policy":"source unchanged; NFC copy with remapped verified spans","device":str(DEVICE),"gpu":torch.cuda.get_device_name(0),"python":platform.python_version(),"libraries":{**{k:installed(k) for k in REQUIRED},"torch":str(torch.__version__)},"train_size":126,"dev_size":37,"test_loaded":False}
if RUN_M3:
    assert E1_DIR.is_dir(), f"Không tìm thấy E1: {E1_DIR}"
    assert all((E1_DIR/k).is_file() and sha256_file(E1_DIR/k)==v for k,v in E1_HASHES.items()), "E1 checkpoint hash mismatch"
    config["e1_checkpoint_sha256"] = E1_HASHES
    config["e1_checkpoint_path"] = str(E1_DIR)
save_json("config.json",config)
print(json.dumps(config,ensure_ascii=False,indent=2))
print("RUN_DIR:",RUN_DIR)

Mounted at /content/drive
{
  "dataset": "VNAB v1.0",
  "dataset_sha256": "e1317a5d27d85426f1e22a0efdb695aacd335a77d79ae9ddd918dd86fa6651ee",
  "model": "deepset/xlm-roberta-base-squad2",
  "model_revision": "a5fab9908c8d856e8c583fd41ba6d92444e46477",
  "run_m3": false,
  "seed": 42,
  "epochs": 3,
  "batch_size": 2,
  "gradient_accumulation": 4,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "optimizer": "AdamW constant LR; clip grad norm 1.0",
  "fp16": true,
  "grad_scaler_init_scale": 1024.0,
  "qa_settings": {
    "max_seq_len": 384,
    "doc_stride": 128,
    "max_answer_len": 128,
    "handle_impossible_answer": false
  },
  "selection": "dev F1, then EM, then earliest epoch; method ties M1 before M2 before M3",
  "metric_normalization": "NFC, lowercase, whitespace; retain punctuation and accents",
  "unicode_policy": "source unchanged; NFC copy with remapped verified spans",
  "device": "cuda:0",
  "gpu": "Tesla T4",
  "python": "3.13.16",
  "libraries": {
    "transformer

## 03. Upload đúng vnab-dataset-v1.0.zip
Kiểm tra checksum cả ZIP đã bàn giao, schema, span, Full QA/MRC, overlap, split manifest. Không upload pilot hay test.

In [4]:
try:
    from google.colab import files
except ImportError:
    files = None

research_dir = next(
    (p for p in (Path("research"), Path.cwd()) if (p / "handoff").is_dir()),
    None,
)

def upload_zip(expected):
    if research_dir is not None:
        matches = list((research_dir / "handoff").glob(f"*/{expected}"))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise ValueError(f"Có nhiều ZIP tên {expected} trong thư mục bàn giao.")
    p = Path(expected)
    if not p.exists():
        if files is None:
            raise FileNotFoundError(f"Không tìm thấy {expected} trong repo hoặc thư mục hiện tại.")
        uploaded = files.upload()
        if expected not in uploaded:
            raise ValueError(f"Hãy upload đúng file {expected}")
        p.write_bytes(uploaded[expected])
    return p

def load_package(path, expected_counts):
    out = {}
    with zipfile.ZipFile(path) as z:
        checks = z.read("checksums.sha256").decode("utf-8").splitlines()
        for line in checks:
            sha, name = line.split(maxsplit=1)
            assert hashlib.sha256(z.read(name)).hexdigest() == sha, f"Checksum sai: {name}"
        for kind in ["qa", "mrc"]:
            schema = json.loads(z.read(f"{kind}.schema.json"))
            validator = jsonschema.Draft202012Validator(schema)
            out[kind] = {}
            for split in ["train", "dev"]:
                name = f"{kind}_{split}.jsonl"
                records = [json.loads(s) for s in z.read(name).decode("utf-8-sig").splitlines() if s.strip()]
                assert len(records) == expected_counts[kind][split], name
                for row in records:
                    validator.validate(row)
                    assert row["review_status"] == "verified", row["id"]
                out[kind][split] = records
            a,b = out[kind]["train"],out[kind]["dev"]
            for split, recs in out[kind].items():
                assert len({r["id"] for r in recs}) == len(recs), (kind,split,"duplicate ID")
            for key in ["id","group_id","context","question"]:
                assert not ({r[key] for r in a} & {r[key] for r in b}), (kind,key)
        for key in ["mapping_ids","old_unit_ids","new_unit_ids"]:
            a,b = out["qa"]["train"],out["qa"]["dev"]
            assert not ({x for r in a for x in r[key]} & {x for r in b for x in r[key]}), key
        for split in ["train","dev"]:
            lookup = {r["id"]:r for r in out["qa"][split]}
            for row in out["mrc"][split]:
                texts, starts = row["answers"]["text"],row["answers"]["answer_start"]
                assert len(texts) == len(starts) > 0
                for text,start in zip(texts,starts):
                    assert 0 <= start and row["context"][start:start+len(text)] == text, row["id"]
                q = lookup[row["id"]]
                for key in ["question","context","group_id","province","type","source_doc","article_clause","context_origin","answerable","review_status"]:
                    assert row[key] == q[key], (row["id"],key)
                assert (q["answer_text"],q["answer_start"]) in list(zip(texts,starts))
        manifest = pd.read_csv(io.BytesIO(z.read("split_manifest.csv")),dtype=str)
        for split in ["train","dev"]:
            assert set(manifest.loc[manifest["split"]==split,"id"]) == {r["id"] for r in out["qa"][split]}, split
    return out


FULL_PATH = upload_zip("vnab-dataset-v1.0.zip")
assert sha256_file(FULL_PATH)==DATASET_SHA256, "ZIP khác bản v1.0 đã kiểm chứng; dừng để rà soát"
full = load_package(FULL_PATH,{"qa":{"train":199,"dev":50},"mrc":{"train":126,"dev":37}})
train_rows=full["mrc"]["train"]
dev_rows=full["mrc"]["dev"]
for split,rows in full["mrc"].items():
    assert all(r["type"]=="T1" and r["answerable"] is True for r in rows)
    print(split,len(rows),"MRC | provinces:",len({r["province"] for r in rows}))
save_json("split_ids.json",{k:[r["id"] for r in v] for k,v in full["mrc"].items()})
print("DATASET OK — train/dev only")

Saving vnab-dataset-v1.0.zip to vnab-dataset-v1.0.zip
train 126 MRC | provinces: 8
dev 37 MRC | provinces: 8
DATASET OK — train/dev only


## 04. Load tokenizer và kiểm tra toàn bộ span
Dùng chính hàm make_feature của notebook 02 đã sửa. Giữ nguyên source và gold offset; quy đổi NFC trên bản sao. Yêu cầu 0 mismatch, nếu khác dừng.

In [5]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,use_fast=True)
assert tokenizer.is_fast and tokenizer.padding_side=="right"
MAX_LENGTH = 384

def norm_boundary(s):
    return unicodedata.normalize("NFC", s).strip()

def make_feature(row, split):
    raw_context = row["context"]
    ctx = unicodedata.normalize("NFC", raw_context)
    question = unicodedata.normalize("NFC", row["question"])
    encoded = tokenizer(question, ctx, truncation=False, return_offsets_mapping=True)
    offsets = encoded["offset_mapping"]
    seq = encoded.sequence_ids()
    if len(encoded["input_ids"]) > MAX_LENGTH:
        raise ValueError(f"{row['id']}: > {MAX_LENGTH} tokens; cần chia cửa sổ, không truncate âm thầm.")
    valid = [i for i,(a,b) in enumerate(offsets) if seq[i]==1 and b>a]
    candidates = []
    for raw_text, raw_start in zip(row["answers"]["text"], row["answers"]["answer_start"]):
        raw_end = raw_start + len(raw_text)
        start = len(unicodedata.normalize("NFC", raw_context[:raw_start]))
        end = len(unicodedata.normalize("NFC", raw_context[:raw_end]))
        text = unicodedata.normalize("NFC", raw_text)
        if ctx[start:end] != text:
            raise ValueError(f"{row['id']}: span không khớp sau khi quy đổi sang NFC.")
        # Chọn các token context phủ ký tự đầu và cuối của gold span.
        left = [i for i in valid if offsets[i][0] <= start < offsets[i][1]]
        right = [i for i in valid if offsets[i][0] < end <= offsets[i][1]]
        for si in left:
            for ei in right:
                if si > ei:
                    continue
                a,b = offsets[si][0], offsets[ei][1]
                if not (a <= start < end <= b):
                    continue
                extracted = ctx[a:b]
                match = norm_boundary(extracted) == norm_boundary(text)
                candidates.append((not match, (start-a)+(b-end), si, ei, extracted, raw_text, raw_start, raw_end, start, end))
    if not candidates:
        raise ValueError(f"{row['id']}: không tìm được token span hợp lệ.")
    mismatch, _, si, ei, extracted, gold, raw_start, raw_end, start, end = min(candidates)
    feature = {k:encoded[k] for k in tokenizer.model_input_names if k in encoded}
    feature.update(start_positions=si,end_positions=ei)
    audit = {
        "id":row["id"], "split":split, "province":row["province"], "tokens":len(encoded["input_ids"]),
        "gold":gold, "answer_start":raw_start, "answer_end_exclusive":raw_end,
        "model_answer_start_nfc":start, "model_answer_end_nfc_exclusive":end,
        "start_token":si, "end_token_inclusive":ei,
        "token_span_text":extracted, "boundary_match_nfc_trim":not mismatch
    }
    return feature,audit

def tokenize_package(package):
    feats, audits = {}, []
    for split, records in package["mrc"].items():
        feats[split] = []
        for row in records:
            feat,audit = make_feature(row,split)
            feats[split].append(feat)
            audits.append(audit)
    return feats,pd.DataFrame(audits)


features, token_audit = tokenize_package(full)
assert (len(features["train"]),len(features["dev"]))==(126,37)
assert token_audit.boundary_match_nfc_trim.all(), "Boundary mismatch: dừng rà soát"
assert ((token_audit.end_token_inclusive-token_audit.start_token+1)<=QA_SETTINGS["max_answer_len"]).all()
token_audit.to_csv(RUN_DIR/"token_audit.csv",index=False,encoding="utf-8-sig")
train_features=features["train"]
print("Train/dev tokenized:",len(train_features),len(features["dev"]))
print("Max tokens:",token_audit.tokens.max(),"Boundary mismatches:",int((~token_audit.boundary_match_nfc_trim).sum()))
print("TOKENIZATION OK")
def collate(rows):
    batch=tokenizer.pad([{k:v for k,v in r.items() if k not in ("start_positions","end_positions")} for r in rows],padding=True,return_tensors="pt")
    for k in ("start_positions","end_positions"): batch[k]=torch.tensor([r[k] for r in rows],dtype=torch.long)
    return batch

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Train/dev tokenized: 126 37
Max tokens: 118 Boundary mismatches: 0
TOKENIZATION OK


## 05. Baseline E2-M1 và hàm đánh giá
Metric giống E1; max qua các gold nếu có nhiều. Cùng NFC đầu vào. Latency chỉ mô tả, không kết luận nhanh hơn khi chưa benchmark warm-up chuẩn.

In [6]:
def nfc(s): return unicodedata.normalize("NFC",s)
def normalize_answer(s): return " ".join(nfc(s).lower().split())
def score_one(pred,gold):
    p,g=normalize_answer(pred),normalize_answer(gold)
    pt,gt=p.split(),g.split()
    em=int(p==g)
    if not pt or not gt: return em,float(pt==gt)
    common=sum((Counter(pt)&Counter(gt)).values())
    return em,2*common/(len(pt)+len(gt))
assert score_one("Sài Gòn","phường Sài Gòn")== (0,0.8)
assert score_one("phường Sài Gòn.","phường Sài Gòn")[0]==0


def load_reader(source,revision=None):
    kwargs={"revision":revision} if revision else {}
    reader,info=AutoModelForQuestionAnswering.from_pretrained(source,output_loading_info=True,**kwargs)
    assert not info["missing_keys"] and not info["mismatched_keys"],info
    return reader.to(DEVICE)
def evaluate(reader,data,tag,tok=None):
    tok=tokenizer if tok is None else tok
    reader.eval()
    qa=pipeline("question-answering",model=reader,tokenizer=tok,device=0)
    records=[]
    for row in tqdm(data,desc=tag):
        torch.cuda.synchronize(); tic=time.perf_counter()
        result=qa(question=nfc(row["question"]),context=nfc(row["context"]),**QA_SETTINGS)
        torch.cuda.synchronize(); ms=(time.perf_counter()-tic)*1000
        pred=result["answer"]; scores=[score_one(pred,g) for g in row["answers"]["text"]]
        records.append({**{k:row[k] for k in ("id","group_id","province","type","source_doc","article_clause","question")},"gold":row["answers"]["text"][0],"gold_answers":json.dumps(row["answers"]["text"],ensure_ascii=False),"prediction":pred,"score":float(result["score"]),"pred_start_nfc":result["start"],"pred_end_nfc_exclusive":result["end"],"em":max(s[0] for s in scores),"f1":max(s[1] for s in scores),"latency_ms":ms})
    df=pd.DataFrame(records)
    assert len(df)==len(data) and df.id.is_unique
    met={"tag":tag,"samples":len(df),"EM_percent":float(df.em.mean()*100),"F1_percent":float(df.f1.mean()*100),"avg_latency_ms":float(df.latency_ms.mean())}
    df.to_csv(RUN_DIR/f"{tag}_predictions.csv",index=False,encoding="utf-8-sig")
    save_json(f"{tag}_metrics.json",met)
    del qa
    return met
model=load_reader(MODEL_NAME,MODEL_REVISION)
config["parameters"]=sum(p.numel() for p in model.parameters())
save_json("config.json",config)
baseline=evaluate(model,dev_rows,"M1_dev")
print(baseline)
del model; gc.collect(); torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Device set to use cuda:0


M1_dev:   0%|          | 0/37 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'tag': 'M1_dev', 'samples': 37, 'EM_percent': 0.0, 'F1_percent': 54.506233035644804, 'avg_latency_ms': 31.333369783785532}


## 06. Fine-tune E2-M2 (và định nghĩa hàm dùng lại cho M3)
Cùng optimizer và gradient accumulation của E1 đã kiểm chứng. Lưu best checkpoint ngay sau từng epoch trên Drive, không ghi đè lượt khác. 126 feature → 16 bước/epoch, 48 bước dự kiến. Không tự động resume optimizer.

In [7]:
def train_method(method,source,revision=None):
    BEST_DIR=RUN_DIR/method/"best_checkpoint"
    assert not BEST_DIR.exists(), "Checkpoint đã tồn tại; không ghi đè lượt cũ."
    model=load_reader(source,revision)
    assert not BEST_DIR.exists(), "Checkpoint đã tồn tại; không ghi đè lượt cũ."
    set_seed(SEED)
    generator=torch.Generator().manual_seed(SEED)
    loader=DataLoader(train_features,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate,generator=generator,num_workers=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_FP16,
        init_scale=INIT_SCALE
    )
    initial_head=model.qa_outputs.weight.detach().cpu().clone()
    training_log=[]; history=[]; best_key=None; best_epoch=None; updates=0
    for epoch in range(1,EPOCHS+1):
        model.train(); optimizer.zero_grad(set_to_none=True)
        group_loss=0.0; group_examples=0; group_batches=0
        # Gom microbatch, chuẩn hóa gradient theo số mẫu trong nhóm (kể cả nhóm cuối).
        for i,batch in enumerate(tqdm(loader,desc=f"Train epoch {epoch}"),start=1):
            batch={k:v.to(DEVICE) for k,v in batch.items()}
            n=batch["input_ids"].shape[0]
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=USE_FP16):
                out=model(**batch); loss=out.loss
            raw=float(loss.detach().item())
            if not math.isfinite(raw): raise RuntimeError(f"Non-finite loss: epoch={epoch}, batch={i}")
            scaler.scale(loss*n).backward()
            group_loss+=raw*n; group_examples+=n; group_batches+=1
            del out,loss,batch
            if i%ACCUM_STEPS==0 or i==len(loader):
                scaler.unscale_(optimizer)
                for p in model.parameters():
                    if p.grad is not None: p.grad.div_(group_examples)
                grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scale_before=scaler.get_scale(); scaler.step(optimizer); scaler.update()
                skipped=scaler.get_scale()<scale_before
                if not skipped: updates+=1
                training_log.append({"epoch":epoch,"batch_end":i,"optimizer_updates":updates,"loss":group_loss/group_examples,"gradient_norm":float(grad_norm) if torch.isfinite(grad_norm) else None,"skipped":bool(skipped),"features":group_examples,"scale_before":scale_before,"scale_after":scaler.get_scale()})
                optimizer.zero_grad(set_to_none=True)
                group_loss=0.0;group_examples=0;group_batches=0
        pd.DataFrame(training_log).to_csv(RUN_DIR/f"{method}_training_log.csv",index=False)
        assert updates>0, "Chưa có optimizer step thành công"
        met=evaluate(model,dev_rows,f"{method}_epoch_{epoch}_dev")
        history.append({"epoch":epoch,**met})
        save_json(f"{method}_dev_history.json",history)
        key=(met["F1_percent"],met["EM_percent"])
        if best_key is None or key>best_key:
            best_key=key;best_epoch=epoch
            model.save_pretrained(BEST_DIR,safe_serialization=True)
            tokenizer.save_pretrained(BEST_DIR)
            save_json(f"{method}_best_selection.json",{"epoch":epoch,"metrics":met,"criterion":config["selection"]})
        print("Epoch",epoch,"dev:",met,"best epoch:",best_epoch)
    head_delta=float((model.qa_outputs.weight.detach().cpu()-initial_head).abs().max())
    assert head_delta>0 and updates>0
    save_json(f"{method}_training_status.json",{"optimizer_updates":updates,"head_max_abs_change_final":head_delta,"best_epoch":best_epoch,"epochs_completed":EPOCHS})
    print("TRAINING OK — optimizer updates:",updates,"QA head max abs change:",head_delta)
    status={"optimizer_updates":updates,"expected_updates":math.ceil(len(loader)/ACCUM_STEPS)*EPOCHS,"skipped_updates":sum(x["skipped"] for x in training_log),"best_epoch":best_epoch,"epochs_completed":EPOCHS,"head_max_abs_change_final":head_delta}
    save_json(f"{method}_training_status.json",status)
    del optimizer,scaler,model
    if "p" in locals(): del p
    gc.collect(); torch.cuda.empty_cache()
    reader=load_reader(BEST_DIR)
    best=evaluate(reader,dev_rows,f"{method}_best_dev")
    selected=next(x for x in history if x["epoch"]==best_epoch)
    assert abs(best["F1_percent"]-selected["F1_percent"])<1e-6 and abs(best["EM_percent"]-selected["EM_percent"])<1e-6, "Reload metrics differ: investigate"
    del reader; gc.collect(); torch.cuda.empty_cache()
    print(method,status,best)
    return {"method":method,"checkpoint":str(BEST_DIR),"best_epoch":best_epoch,**best}
m2=train_method("M2",MODEL_NAME,MODEL_REVISION)

Train epoch 1:   0%|          | 0/63 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Device set to use cuda:0


M2_epoch_1_dev:   0%|          | 0/37 [00:00<?, ?it/s]

Epoch 1 dev: {'tag': 'M2_epoch_1_dev', 'samples': 37, 'EM_percent': 0.0, 'F1_percent': 67.34234234234235, 'avg_latency_ms': 11.399599729726118} best epoch: 1


Train epoch 2:   0%|          | 0/63 [00:00<?, ?it/s]

Device set to use cuda:0


... (5 outputs left out)

Epoch 3 dev: {'tag': 'M2_epoch_3_dev', 'samples': 37, 'EM_percent': 0.0, 'F1_percent': 67.34234234234235, 'avg_latency_ms': 17.450562999997416} best epoch: 1
TRAINING OK — optimizer updates: 48 QA head max abs change: 0.0006238259375095367


Device set to use cuda:0


M2_best_dev:   0%|          | 0/37 [00:00<?, ?it/s]

M2 {'optimizer_updates': 48, 'expected_updates': 48, 'skipped_updates': 0, 'best_epoch': 1, 'epochs_completed': 3, 'head_max_abs_change_final': 0.0006238259375095367} {'tag': 'M2_best_dev', 'samples': 37, 'EM_percent': 0.0, 'F1_percent': 67.34234234234235, 'avg_latency_ms': 14.38974932432763}


## 07. E2-M3 tùy chọn và E1-only diagnostic
Nếu RUN_M3=False, cell này bỏ qua. Nếu True, kiểm tra tokenizer checkpoint E1 tương đương tokenizer gốc trên toàn bộ train/dev trước khi dùng chung. E1-only là đối chứng bổ sung, không dùng để chọn phương pháp chính.

In [8]:
m3=None; e1_only=None
if RUN_M3:
    e1_tokenizer=AutoTokenizer.from_pretrained(E1_DIR,use_fast=True)
    assert e1_tokenizer.is_fast and e1_tokenizer.get_vocab()==tokenizer.get_vocab()
    for row in train_rows+dev_rows:
        args=(nfc(row["question"]),nfc(row["context"]))
        kwargs={"truncation":False,"return_offsets_mapping":True}
        x,y=tokenizer(*args,**kwargs),e1_tokenizer(*args,**kwargs)
        assert x["input_ids"]==y["input_ids"] and x["offset_mapping"]==y["offset_mapping"],row["id"]
    model=load_reader(E1_DIR)
    e1_only=evaluate(model,dev_rows,"E1_only_dev")
    del model; gc.collect(); torch.cuda.empty_cache()
    m3=train_method("M3",E1_DIR)
else:
    print("SKIP M3 — optional experiment disabled")

SKIP M3 — optional experiment disabled


## 08. So sánh dev và phân tích theo tỉnh
Không mặc định fine-tune tốt hơn. Chọn phương pháp triển khai bằng F1 → EM → ưu tiên đơn giản. E1-only diagnostic báo cáo riêng. Tất cả MRC dev thuộc T1; kết quả chưa phải test độc lập.

In [9]:
methods=[{"method":"M1","checkpoint":MODEL_NAME,"best_epoch":None,**baseline},m2]
if m3 is not None: methods.append(m3)
comparison=pd.DataFrame(methods)
comparison.to_csv(RUN_DIR/"dev_comparison.csv",index=False,encoding="utf-8-sig")
display(comparison[["method","samples","EM_percent","F1_percent","best_epoch"]])
if e1_only is not None: print("E1-only diagnostic:",e1_only)
selected=max(methods,key=lambda x:(x["F1_percent"],x["EM_percent"],-int(x["method"][1:])))
print("Selected on dev:",selected["method"])
by_province=[]
for m in methods:
    df=pd.read_csv(RUN_DIR/f'{m["tag"]}_predictions.csv')
    assert list(df.id)==[r["id"] for r in dev_rows]
    for prov,g in df.groupby("province"):
        by_province.append({"method":m["method"],"province":prov,"samples":len(g),"EM_percent":g.em.mean()*100,"F1_percent":g.f1.mean()*100})
pd.DataFrame(by_province).to_csv(RUN_DIR/"dev_by_province.csv",index=False,encoding="utf-8-sig")
print("DEV COMPARISON OK")

,method,samples,EM_percent,F1_percent,best_epoch
0,M1,37,0.0,54.506233,NaN
1,M2,37,0.0,67.342342,1.0


Selected on dev: M2
DEV COMPARISON OK


## 09. Xuất kết quả dev và manifest dự kiến
Checkpoint đã nằm trong Drive. ZIP chỉ gồm báo cáo, không chứa trọng số. Gửi notebook có output và ZIP này để rà trước khi khóa.

In [10]:
def checkpoint_hashes(folder):
    folder=Path(folder)
    assert (folder/"model.safetensors").is_file()
    return {str(p.relative_to(folder)):sha256_file(p) for p in sorted(folder.rglob("*")) if p.is_file()}
artifacts={"M1":{"model":MODEL_NAME,"revision":MODEL_REVISION}}
for m in methods[1:]:
    artifacts[m["method"]]={"path":m["checkpoint"],"best_epoch":m["best_epoch"],"sha256":checkpoint_hashes(m["checkpoint"])}
manifest={"locked":False,"test_loaded":False,"config":config,"methods":methods,"selected_method":selected["method"],"artifacts":artifacts,"diagnostic_e1_only":e1_only,"scope":"MRC given context; VNAB dev T1 only"}
save_json("selection_manifest.json",manifest)
def export_report(suffix):
    report_dir=Path("/content")/(RUN_DIR.name+"_"+suffix+"_export")
    report_dir.mkdir(exist_ok=False)
    for p in RUN_DIR.iterdir():
        if p.is_file() and p.suffix in (".json",".csv"): shutil.copy2(p,report_dir/p.name)
    archive=shutil.make_archive(str(Path("/content")/(RUN_DIR.name+"_"+suffix)),"zip",root_dir=report_dir)
    files.download(archive)
    print("RESULTS SAVED:",archive)
    return archive
archive=export_report("dev_results")
print("Checkpoints and logs:",RUN_DIR)
print("DEV COMPLETE — chưa khóa, chưa test")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RESULTS SAVED: /content/vnab_20261127_151306_dev_results.zip
Checkpoints and logs: /content/drive/MyDrive/VNAB_research/E2_runs/vnab_20261127_151306
DEV COMPLETE — chưa khóa, chưa test


## 10. Khóa sau khi rà soát (không chạy test)
Chỉ đổi CONFIRM_LOCKED=True khi cả nhóm xác nhận cấu hình, kết quả, checksum và các phương pháp đánh giá. Vi nhận input-only và manifest ở ngày 5; notebook này không tải test.

In [ ]:
CONFIRM_LOCKED=False
if not CONFIRM_LOCKED:
    print("SKIP LOCK — review dev results first")
else:
    locked=json.loads((RUN_DIR/"selection_manifest.json").read_text())
    assert locked["config"]==config and not locked["locked"]
    for key,obj in locked["artifacts"].items():
        if key!="M1": assert checkpoint_hashes(obj["path"])==obj["sha256"],key
    locked["locked"]=True
    locked["locked_at_utc"]=time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())
    save_json("locked_manifest.json",locked)
    locked_archive=export_report("locked_results")
    print("LOCKED — hand off to Vi; no test loaded")